---
title: "ELE618 - Sistemas Elétricos Industriais"
subtitle: "Apostila de Sistemas Elétricos Industriais - José A. Toledo Jr.<br>Seleção da Máquina Elétrica"
author: "Prof. Dr. Thales A. C. Maia"
institute: "UFMG - DEE"
draft: true
engine: julia
execute:
  echo: true
format:
  revealjs:
    code-fold: true
    theme: serif
    css: ../reveal-site-like.css
    slide-number: true
    chalkboard: true
    controls: true
    progress: true
    history: true
    preview-links: auto
    footer: "UFMG | ELE618 - Sistemas Elétricos Industriais"
    transition: slide
    background-transition: fade
    scrollable: false
    margin: 0.05
    slide-level: 2
    navigation-mode: default
    controls-layout: bottom-right
    controls-tutorial: true
    mouse-wheel: true
lang: pt-BR
crossref:
  fig-title: Fig.
  tbl-title: Tab.
---

# Seleção da Máquina Elétrica

## Especificação do Tempo de Partida

O sistema acelera enquanto $T_{\rm{ac}} > 0$. O tempo de aceleração $t_{\rm{ac}}$ (repouso até regime permanente) importa para:

- verificar se o motor consegue acionar a carga;
- dimensionar instalação, partida e proteção;
- calcular disponibilidade de potência mediante partidas/frenagens/reversões.

As normas ABNT NBR 17094 / IEC 60034-1 exigem que o motor suporte 2 partidas sucessivas (a frio + após parar) e 1 partida a quente.

## Expressão Exata do Tempo de Aceleração

Integrando a equação de Newton de $0$ até $t_{\rm{ac}}$:

$$
t_{\rm{ac}} = \int_{0}^{t_{\rm{ac}}} dt = \int_{0}^{\omega_{\rm{op}}} \dfrac{J}{T_{\rm{ac}}(\omega_{\rm{mec}})}\, d\omega_{\rm{mec}}
$$ {#eq-tacIntegral}

$T_{\rm{ac}}(\omega_{\rm{mec}})$ vem da subtração das curvas de torque eletromagnético e de carga — expressá-la analiticamente (e integrar sua inversa) raramente é trivial.

## Método dos Intervalos (Torque Médio)

Divide-se a curva de aceleração em intervalos, usando um torque médio $\overline{T}_{\rm{ac},i}$ em cada um:

![Divisão em intervalos da curva de aceleração.](images/curvaAceleracaoIntervalos.svg){#fig-curvaAceleracaoIntervalos width=55% fig-align='center'}

## Método dos Intervalos (Torque Médio)

$$
\overline{T}_{\rm{ac},i} = \dfrac{T_{i+1} + T_{i}}{2}, \qquad
t_{\rm{ac},i} = \dfrac{J}{\overline{T}_{\rm{ac},i}} (\omega_{i+1} - \omega_{i})
$$ {#eq-tacIntervalo}

$$
t_{\rm{ac}} = \sum_{i} t_{\rm{ac},i}
$$

Mamede Filho recomenda incrementos de 10% da velocidade síncrona, exceto onde o torque varia acentuadamente.

::: aside
**De onde vem a média aritmética?** É a aproximação mais simples para a integral exata (@eq-tacIntegral) dentro de um intervalo pequeno: assume-se que $T_{\rm{ac}}(\omega)$ é aproximadamente **constante**, igual à média dos dois valores nas bordas do intervalo. Quanto menor o intervalo, melhor essa aproximação — por isso a recomendação de incrementos pequenos (10% da velocidade síncrona). O torque efetivo (próximo slide) é uma versão mais precisa da mesma ideia.
:::

## Exemplo: Tempo de Aceleração pelo Método dos Intervalos

Acionamento com $J=0{,}5$ kg.m², velocidade de operação $\omega_{\rm{op}}=180$ rad/s, dividida em 5 intervalos iguais. Torque de aceleração ($T_e - T_L$) em cada ponto:

| $\omega$ (rad/s) | 0 | 36 | 72 | 108 | 144 | 180 |
|---|---|---|---|---|---|---|
| $T_{\rm{ac}}$ (N.m) | 80 | 60 | 50 | 70 | 90 | 40 |

In [1]:
omega = [0.0, 36.0, 72.0, 108.0, 144.0, 180.0]  # rad/s
Tac   = [80.0, 60.0, 50.0, 70.0, 90.0, 40.0]     # N.m
J = 0.5   # kg.m^2

tac_total = 0.0
for k in 1:length(omega)-1
    Tbar = (Tac[k+1] + Tac[k]) / 2
    dt = J / Tbar * (omega[k+1] - omega[k])
    global tac_total += dt
    println("intervalo ", k, ": T_medio = ", round(Tbar, digits=2), " N.m, dt = ", round(dt, digits=4), " s")
end
println("t_ac total = ", round(tac_total, digits=4), " s")

intervalo 1: T_medio = 70.0 N.m, dt = 0.2571 s
intervalo 2: T_medio = 55.0 N.m, dt = 0.3273 s
intervalo 3: T_medio = 60.0 N.m, dt = 0.3 s
intervalo 4: T_medio = 80.0 N.m, dt = 0.225 s
intervalo 5: T_medio = 65.0 N.m, dt = 0.2769 s
t_ac total = 1.3863 s


## Refinamento: Torque de Aceleração Efetivo

Lobosco e Dias (1988) substituem o torque médio por um torque **efetivo**:

$$
T_{\rm{ac,ef},i} = \dfrac{T_{i+1} - T_{i}}{\ln{ \left( \dfrac{T_{i+1}}{T_{i}} \right) }}
$$ {#eq-Tacef}

Método mais preciso, mas com singularidade matemática quando $T_{i+1}=T_i$ ou quando algum torque é nulo.

::: aside
**De onde vem o $\ln$?** Da integral exata (@eq-tacIntegral), assumindo que $T_{\rm{ac}}(\omega)$ varia **linearmente** com $\omega$ dentro do intervalo (de $T_i$ até $T_{i+1}$). Substituindo a variável de integração de $\omega$ para $T$, a integral $\int d\omega / T_{\rm{ac}}(\omega)$ vira $\int dT/T = \ln T$ — daí o logaritmo. O resultado é a **média logarítmica** de $T_i$ e $T_{i+1}$: mais precisa que a média aritmética simples da @eq-tacIntervalo, que assume $T_{\rm{ac}}$ constante (igual ao valor médio) dentro do intervalo, em vez de variando ponto a ponto.
:::

## Exemplo: Torque Efetivo *vs.* Torque Médio

Para um intervalo entre $T_i = 80$ N.m e $T_{i+1} = 60$ N.m, compare o torque médio simples com o torque efetivo de Lobosco:

In [2]:
Ti, Tip1 = 80.0, 60.0

Tmedio = (Ti + Tip1) / 2
Tef = (Tip1 - Ti) / log(Tip1 / Ti)

println("T medio    = ", Tmedio, " N.m")
println("T efetivo  = ", round(Tef, digits=3), " N.m")

T medio    = 70.0 N.m
T efetivo  = 69.521 N.m


A diferença entre os dois métodos é pequena (menos de 1% neste intervalo) — por isso, na prática, o método do torque médio simples costuma ser suficiente, reservando-se o torque efetivo para os casos em que se busca maior precisão.

## Método Simplificado (Torque Médio Único)

Na prática, um único valor médio para todo o acionamento é suficiente:

$$
\overline{T}_{\rm{ac}} = \overline{T}_{\rm{e}} - \overline{T}_{\rm{L}}, \qquad
t_{\rm{ac}} = \dfrac{J~\omega_{\rm{op}}}{\overline{T}_{\rm{ac}}}
$$ {#eq-tacSimplificado}

Para motores de indução de 2 polos, Lobosco e Dias recomendam acrescentar 10% ao tempo calculado (baixo torque mínimo típico desses motores).

![Representação do torque médio de aceleração.](images/curvasConjugadoMedio.svg){#fig-curvasConjugadoMedio width="48%" fig-align='center'}

::: aside
É o mesmo método dos intervalos (@eq-tacIntervalo), levado ao limite de **um único intervalo** cobrindo toda a aceleração (de $0$ a $\omega_{\rm{op}}$) — troca-se precisão por simplicidade de cálculo. Já os 10% extra para motores de 2 polos **são** empíricos: uma margem de segurança de Lobosco e Dias para compensar o torque mínimo tipicamente baixo desses motores, que o método (baseado só nas médias) não capta diretamente.
:::


## Torque Médio Eletromagnético (Aproximação)

Com $T_{\rm{e,p}}$ e $T_{\rm{e,max}}$ (torques de partida e máximo do motor):

$$
\overline{T}_{\rm{e}} =
\begin{cases}
0{,}45 \left( T_{\rm{e,p}} + T_{\rm{e,max}} \right), & \text{categorias N e H;} \\
0{,}60~T_{\rm{e,p}}, & \text{categoria D.}
\end{cases}
$$ {#eq-torqueMedioEletromagnetico}

::: aside
**De onde vêm 0,45 e 0,60?** Ao contrário do $\ln$ do torque efetivo (Lobosco), aqui **não há uma integral fechada por trás** — são coeficientes **empíricos**, ajustados por Augusto Júnior (2005) e Mamede Filho (2017) a partir de curvas reais de torque × velocidade.
:::

- **N/H:** uma média simples seria $0{,}5(T_{\rm{e,p}}+T_{\rm{e,max}})$. O coeficiente real (0,45) é **menor** porque, entre a partida e o torque máximo, a curva desses motores passa por um **torque mínimo** ($T_{\rm{min}}$) — a curva "afunda" no meio do caminho antes de subir até $T_{\rm{max}}$. Isso puxa a média real (pela área sob a curva) para baixo da média ingênua dos dois extremos.
- **D:** motores de categoria D têm curva de torque decrescente da partida até a operação, sem esse afundamento pronunciado — por isso a aproximação usa só $T_{\rm{e,p}}$ (nem precisa de $T_{\rm{e,max}}$), escalado por 0,6.


## Exemplo: Torque Médio Eletromagnético

Motor categoria N com $T_{\rm{e,p}} = 180$ N.m e $T_{\rm{e,max}} = 220$ N.m. Compare com um motor categoria D de mesmo $T_{\rm{e,p}}' = 160$ N.m:

In [3]:
Tep, Temax = 180.0, 220.0   # motor categoria N/H
Tebar_NH = 0.45 * (Tep + Temax)
println("T_e_bar (N/H) = ", Tebar_NH, " N.m")

Tep_D = 160.0               # motor categoria D
Tebar_D = 0.60 * Tep_D
println("T_e_bar (D)   = ", Tebar_D, " N.m")

T_e_bar (N/H) = 180.0 N.m
T_e_bar (D)   = 96.0 N.m


## Correção para Partida com Tensão Reduzida

Com $k = V_{\rm{red}} / V_{\rm{nom}}$:

$$
\overline{T}_{\rm{e}} =
\begin{cases}
0{,}45 \left( T_{\rm{e,p}}\, k^{2{,}2} + T_{\rm{e,max}}\, k^{2{,}0} \right), & \text{categorias N e H;} \\
0{,}60~T_{\rm{e,p}}\, k^{2{,}2}, & \text{categoria D.}
\end{cases}
$$ {#eq-torqueMedioReduzida}

::: aside
**De onde vêm os expoentes $2{,}2$ e $2{,}0$?** A base é física: num motor de indução, a corrente do rotor é proporcional à tensão aplicada ($I \propto V$, circuito linear equivalente), e o torque depende do quadrado dessa corrente ($T \propto I^2$) — logo $T \propto V^2$ em qualquer escorregamento, com frequência fixa.

- **$T_{\rm{e,max}}$ ($k^{2{,}0}$):** segue exatamente essa lei quadrática ideal — o torque máximo é bem descrito pelo circuito equivalente linear mesmo com tensão reduzida.
- **$T_{\rm{e,p}}$ ($k^{2{,}2}$):** aqui o expoente $2{,}2$ (em vez de $2{,}0$) já é **ajuste empírico**, não dedução. Na partida (rotor bloqueado), a corrente é muito mais alta e a máquina satura magneticamente mais — isso faz o torque de partida cair um pouco mais rápido que a lei quadrática ideal quando a tensão é reduzida, daí o expoente levemente maior.
:::

## Exemplo: Partida Estrela-Triângulo

Na ligação estrela-triângulo, o enrolamento vê $1/\sqrt{3} \approx 57{,}7\%$ da tensão de linha durante a partida. Para o motor N/H do exemplo anterior ($T_{\rm{e,p}}=180$, $T_{\rm{e,max}}=220$ N.m):

In [4]:
Tep, Temax = 180.0, 220.0
k = 1 / sqrt(3)
println("k = Vred/Vnom = ", round(k, digits=4))

Tebar_reduzida = 0.45 * (Tep * k^2.2 + Temax * k^2.0)
println("T_e_bar com partida estrela = ", round(Tebar_reduzida, digits=3), " N.m")
println("T_e_bar em tensao plena     = ", 0.45*(Tep+Temax), " N.m")

k = Vred/Vnom = 0.5774
T_e_bar com partida estrela = 57.191 N.m
T_e_bar em tensao plena     = 180.0 N.m


A partida estrela-triângulo reduz o torque médio eletromagnético disponível para menos de **um terço** do valor em tensão plena — por isso ela só é viável quando a carga tem baixo torque de partida.

## Torque Médio de Carga

Com $\beta$ dependendo da natureza da carga (0: constante, 1: linear, 2: quadrática — Seção anterior):

$$
\overline{T}_{\rm{L}} = T_{\rm{L,p}} + \dfrac{\left( T_{\rm{L,op}} - T_{\rm{L,p}} \right)}{\beta + 1}
$$ {#eq-torqueMedioCarga}

Para torque inverso à rotação, o valor médio é problemático (inspeção visual/computacional); para torque não uniforme, é particular a cada aplicação; para cargas sem torque, é nulo.

::: aside
**Esta, ao contrário de várias outras, tem dedução fechada.** Modelando a carga como $T_{\rm{L}}(\omega) = T_{\rm{L,p}} + \alpha_{\rm{L}}\,\omega^{\beta}$, a média integral em $[0,\omega_{\rm{op}}]$ é $T_{\rm{L,p}} + \alpha_{\rm{L}}\,\omega_{\rm{op}}^{\beta}/(\beta+1)$. Como $T_{\rm{L,op}} = T_{\rm{L,p}} + \alpha_{\rm{L}}\,\omega_{\rm{op}}^{\beta}$ (a própria definição de $T_{\rm{L,op}}$), basta substituir $\alpha_{\rm{L}}\,\omega_{\rm{op}}^{\beta} = T_{\rm{L,op}}-T_{\rm{L,p}}$ para chegar na fórmula ao lado — confirmado numericamente por integração direta.
:::

## Exemplo: Torque Médio de Carga

Carga com torque linear com a rotação ($\beta=1$), $T_{\rm{L,p}} = 30$ N.m na partida e $T_{\rm{L,op}} = 90$ N.m no ponto de operação:

In [5]:
TLp, TLop, beta = 30.0, 90.0, 1
TLbar = TLp + (TLop - TLp) / (beta + 1)
println("T_L_bar = ", TLbar, " N.m")

T_L_bar = 60.0 N.m


## Exemplo: Tempo de Aceleração Simplificado e Limite de Rotor Bloqueado

Com $\overline{T}_{\rm{e}} = 180$ N.m, $\overline{T}_{\rm{L}} = 60$ N.m, $J=0{,}5$ kg.m², $\omega_{\rm{op}}=180$ rad/s, e tempo de rotor bloqueado do fabricante $t_{\rm{rb,max}}=12$ s:

In [6]:
Tebar, TLbar = 180.0, 60.0
J, omega_op = 0.5, 180.0

Tacbar = Tebar - TLbar
tac_simpl = J * omega_op / Tacbar
println("t_ac simplificado = ", round(tac_simpl, digits=3), " s")

trb_max = 12.0
limite = 0.8 * trb_max
println("limite 0,8 * t_rb_max = ", limite, " s -> ", tac_simpl <= limite ? "atende" : "excede")

t_ac simplificado = 0.75 s
limite 0,8 * t_rb_max = 9.600000000000001 s -> atende


## Classe de Isolamento e Vida Útil

$$
t_{\rm{ac}} \leq 0{,}8~t_{\rm{rb,max}}
$$

Se o motor não atinge $\omega_{\rm{op}}$ nesse prazo, a corrente de estator (6 a 9 vezes a nominal) aquece perigosamente o enrolamento.

| Classe de isolamento | A | E | B | F | H |
|---|---|---|---|---|---|
| Temperatura ambiente (°C) | 40 | 40 | 40 | 40 | 40 |
| Elevação de temperatura $\Delta t$ (°C) | 60 | 75 | 80 | 105 | 125 |
| Diferença ponto mais quente / média (°C) | 5 | 5 | 10 | 10 | 15 |
| Temperatura total (°C) | 105 | 120 | 130 | 155 | 180 |

Um aumento de 8 a 10 °C acima do limite da classe pode reduzir a vida útil do enrolamento pela metade.

::: aside
Os $0{,}8$ do limite de rotor bloqueado e todos os valores da tabela são **normativos/empíricos** — não vêm de uma fórmula, vêm de normas (ABNT/IEC) e de dados de degradação de materiais isolantes reais (o verniz isolante envelhece e perde a capacidade de isolar em função da temperatura e do tempo de exposição — uma relação estudada empiricamente e resumida nesses limites por classe).
:::

## Especificação da Potência Mecânica

Potência de um agente que desenvolve torque: $P(t) = T(t)\,\omega(t)$. Multiplicando a Eq. de Newton por $\omega_{\rm{mec}}(t)$:

$$
\underbrace{T_{\rm{e}}(t) \omega_{\rm{mec}}(t)}_{P_{\rm{mec}}(t)} = \underbrace{T_{\rm{L}}(t) \omega_{\rm{mec}}(t)}_{P_{\rm{L}}(t)} + \underbrace{J \frac{d\omega_{\rm{mec}}(t)}{dt} \omega_{\rm{mec}}(t)}_{P_{\rm{ac}}(t)}
$$ {#eq-relacoesPotencia}

![Potência mecânica a partir do torque e da velocidade angular.](images/graficoTorqueVelocidadePotencia.png){#fig-graficoTorqueVelocidadePotencia .r-stretch fig-align='center'}

## Potência de Pico *vs.* Potência Eficaz

Selecionar pelo pico (≈ 42 kW / 57 cv) garante a reprodução da forma de onda, mas **superdimensiona** o motor. Como o motor suporta sobrecarga limitada, uma potência nominal menor pode bastar — desde que calculada pela **potência eficaz** (mesmas perdas/elevação de temperatura que a potência cíclica variável):

$$
P_{\rm{ef}} = \sqrt{ \dfrac{1}{T} \int^{T}_{0} P_{\rm{mec}}(t)^{2}\, dt }
$$ {#eq-PefIntegral}

Para o ciclo da @fig-graficoTorqueVelocidadePotencia, $P_{\rm{ef}} = 26{,}8$ kW — o que leva à escolha de um motor comercial de **40 cv**, em vez dos 60 cv do valor de pico.

::: aside
**Por que essa raiz-da-média-dos-quadrados (RMS)?** É a mesma lógica da corrente RMS em circuitos elétricos: as perdas por efeito Joule crescem com o **quadrado** da corrente (e, numa faixa de operação razoável, com o quadrado da potência). Uma potência constante $P_{\rm{ef}}$ causa o mesmo aquecimento médio que a potência cíclica variável real quando ambas têm o mesmo valor médio quadrático — daí elevar ao quadrado, integrar/tirar a média, e depois tirar a raiz.
:::

## Regimes de Serviço

Com o ciclo conhecido, especifica-se o regime de serviço com precisão — 10 regimes padronizados pela NBR 17094-1 (S1 a S10):

:::: {.columns}
::: {.column width="50%"}
![Funcionamento contínuo com solicitações intermitentes.](images/cicloContinuoSolicitacaoIntermitente.svg){#fig-cicloContinuo width=100% fig-align='center'}
:::
::: {.column width="50%"}
![Carga variável com repouso entre os tempos de carga.](images/cicloVariavelRepouso.svg){#fig-cicloVariavel width=100% fig-align='center'}
:::
::::

## Regimes S1 (Contínuo) e S8

:::: {.columns}
::: {.column width="45%"}
![Regime S1: carga constante até equilíbrio térmico.](images/regimeS1.svg){#fig-regimeS1 width=100% fig-align='center'}
:::
::: {.column width="55%"}
![Regime S8: funcionamento contínuo com mudança periódica carga/velocidade, sem repouso.](images/regimeS8.svg){#fig-regimeS8 width=100% fig-align='center'}
:::
::::

Em nenhum dos 10 regimes a temperatura do motor ultrapassa $\theta_{\rm{max}}$ da isolação.

## Simplificações Práticas

- **Carga de pequena inércia:** $P_{\rm{mec}}(t) \approx P_{\rm{L}}(t)$ (potência de aceleração desprezível);
- **Cargas constantes por trechos** (comum na indústria) simplifica a integral em um somatório:

$$
P_{\rm{ef}} = \sqrt{ \dfrac{ \sum P_{\rm{L},i}(t)^{2} \cdot t_{i} }{ \sum t_{i} } }
$$ {#eq-PefSemRepouso}

Válida para motores que giram continuamente (sem repouso entre cargas).

::: aside
Sem novidade matemática: é a @eq-PefIntegral discretizada — quando a carga é constante por trechos, a integral $\int P(t)^2\,dt$ vira exatamente uma soma de retângulos $\sum P_{L,i}^2\,t_i$, sem nenhuma aproximação extra.
:::

## Exemplo: Potência Eficaz sem Repouso

Ciclo de carga com 3 patamares:

| Patamar | $P_L$ (kW) | Duração (s) |
|---|---|---|
| 1 | 10 | 20 |
| 2 | 25 | 30 |
| 3 | 15 | 10 |

In [8]:
PLi = [10.0, 25.0, 15.0]   # kW
ti  = [20.0, 30.0, 10.0]   # s

Pef = sqrt(sum(PLi.^2 .* ti) / sum(ti))
println("P_ef = ", round(Pef, digits=3), " kW  (pico = ", maximum(PLi), " kW)")

P_ef = 19.579 kW  (pico = 25.0 kW)


## Ciclo com Repouso ($k_{\rm{v}}$)

Motores autoventilados esfriam pior parados. Com tempo de repouso $t_{\rm{r}}$ e $k_{\rm{v}}$ (1: ventilação independente do funcionamento — 3: ventilação vinculada ao funcionamento):

$$
P_{\rm{ef}} = \sqrt{ \dfrac{ \sum P_{\rm{L},i}(t)^{2} \cdot t_{i} }{ \sum t_{i} + \dfrac{t_{r}}{k_{\rm{v}}} } }
$$ {#eq-PefComRepouso}

::: aside
O termo $t_r/k_{\rm{v}}$ é uma **correção empírica**, não uma dedução: fisicamente, o repouso deveria contar como tempo "de graça" para resfriar (reduzindo a potência efetiva necessária), mas só na medida em que a ventilação continue funcionando parada. $k_{\rm{v}}$ pondera esse efeito — quanto maior $k_{\rm{v}}$ (motor autoventilado, ventilação para junto com o eixo), menos o repouso ajuda a resfriar, e por isso menos ele "conta" no denominador.
:::

## Exemplo: Potência Eficaz com Repouso

Ao ciclo de 3 patamares, acrescente um repouso de $t_r = 15$ s, com motor autoventilado ($k_v = 3$):

In [9]:
PLi = [10.0, 25.0, 15.0]
ti  = [20.0, 30.0, 10.0]
tr, kv = 15.0, 3.0

Pef_com_repouso = sqrt(sum(PLi.^2 .* ti) / (sum(ti) + tr/kv))
println("P_ef (com repouso) = ", round(Pef_com_repouso, digits=3), " kW")
println("P_ef (sem repouso, referencia) = ", round(sqrt(sum(PLi.^2 .* ti)/sum(ti)), digits=3), " kW")

P_ef (com repouso) = 18.811 kW
P_ef (sem repouso, referencia) = 19.579 kW


Com repouso, $P_{\rm{ef}}$ **cai** de 19,6 kW para 18,8 kW: o mesmo trabalho térmico fica distribuído em um ciclo de tempo total maior, permitindo (a princípio) um motor comercial menor. O fator $k_{\rm{v}}=3$ já penaliza esse ganho — motores autoventilados (TEFC) esfriam pouco parados, então o repouso conta menos como "tempo de resfriamento" do que contaria para um motor com ventilação forçada ($k_{\rm{v}}=1$).

## Condições de Torque do Motor Selecionado

Além da potência, o motor deve satisfazer:

$$
T_{\rm{e,p}} > T_{\rm{L,p}}, \qquad T_{\rm{e,max}} > 1{,}2 \cdot T_{\rm{L,max}}
$$

(regra prática: torque máximo do motor pelo menos 20% acima do maior torque resistente).

::: aside
A primeira condição ($T_{\rm{e,p}} > T_{\rm{L,p}}$) é lógica pura: sem isso o motor nem sai do repouso. Já a margem de **20%** na segunda condição é uma **regra prática** de Augusto Júnior — uma folga de segurança para picos de carga, quedas momentâneas de tensão e a própria tolerância normativa de $-10\%$ no torque máximo medido (Tab. de tolerâncias, mais adiante), não um valor deduzido analiticamente.
:::

## Exemplo: Verificação das Condições de Torque

Motor com $T_{\rm{e,p}}=180$ N.m e $T_{\rm{e,max}}=220$ N.m, para uma carga com $T_{\rm{L,p}}=140$ N.m e $T_{\rm{L,max}}=170$ N.m:

In [10]:
Tep, TLp = 180.0, 140.0
Temax, TLmax = 220.0, 170.0

println("T_e,p > T_L,p ?        ", Tep > TLp, "   (", Tep, " > ", TLp, ")")
println("T_e,max > 1,2 T_L,max ? ", Temax > 1.2*TLmax, "  (", Temax, " > ", round(1.2*TLmax, digits=1), ")")

T_e,p > T_L,p ?        true   (180.0 > 140.0)
T_e,max > 1,2 T_L,max ? true  (220.0 > 204.0)


## Redução de Potência: Partidas, Frenagens e Reversões

Perdas Joule na partida/frenagem/reversão elevam a temperatura. Lobosco e Dias: uma frenagem equivale a 3 partidas; uma reversão elétrica, a 4 partidas.

$$
P_{\rm{disp,pfr}} = \underbrace{ \sqrt{ \dfrac{ 3600 - k_{\rm{pfr}} \cdot N_{\rm{p}} \cdot t_{\rm{ac}} \cdot \left( \dfrac{I_{\rm{p}}}{I_{\rm{nom}}} \right)^{2} }{ 3600 - 2 \cdot N_{\rm{p}} \cdot t_{\rm{ac}} } } }_{\alpha_{\rm{pfr}}} \cdot P_{\rm{nom}}
$$ {#eq-PdispPFR}

$k_{\rm{pfr}}=1$ (partida), $3$ (frenagem) ou $4$ (reversão); $N_{\rm{p}}$: repetições/hora.

::: aside
Fórmula **empírica** de Lobosco e Dias. O $3600$ é só a conversão de unidades (segundos numa hora, já que $N_{\rm{p}}$ é em repetições/hora e $t_{\rm{ac}}$ em segundos) — isso é dimensional, não empírico. Mas os multiplicadores $k_{\rm{pfr}}=3$ (frenagem) e $4$ (reversão) **são** estimativas empíricas dos autores para o efeito térmico equivalente de cada operação comparado a uma partida simples — não vêm de um cálculo termodinâmico fechado, são uma simplificação prática validada por eles com dados de campo.
:::

## Exemplo: Potência Disponível com Partidas Frequentes

Motor de $P_{\rm{nom}} = 50$ cv, com $N_p = 4$ partidas/hora, $t_{\rm{ac}} = 8$ s, e $I_p/I_{\rm{nom}} = 7$:

In [11]:
kpfr = 1.0     # partida
Np = 4.0       # partidas por hora
tac_h = 8.0    # s
Ip_Inom = 7.0
Pnom = 50.0    # cv

alpha_pfr = sqrt((3600 - kpfr*Np*tac_h*(Ip_Inom)^2) / (3600 - 2*Np*tac_h))
Pdisp_pfr = alpha_pfr * Pnom

println("alpha_pfr = ", round(alpha_pfr, digits=4))
println("P_disp,pfr = ", round(Pdisp_pfr, digits=2), " cv")

alpha_pfr = 0.7581
P_disp,pfr = 37.9 cv


Mesmo com apenas 4 partidas por hora, o fator $\alpha_{\rm{pfr}}$ reduz a potência disponível para cerca de 76% da nominal — o efeito cresce rapidamente com $N_p$, $t_{\rm{ac}}$ e, principalmente, com o quadrado da relação $I_p/I_{\rm{nom}}$.

## Redução por Ambiente Atípico

Norma exige potência nominal sem sobreaquecimento até 1000 m de altitude e 40 °C. Acima disso, o ar rarefeito piora o arrefecimento:

$$
P_{\rm{disp,amb}} = \alpha_{\rm{amb}} \cdot P_{\rm{nom}}
$$ {#eq-PdispAmb}

| $T_{\rm{amb}}$ (°C) / $H$ (m) | 1000 | 2000 | 3000 | 4000 |
|---|---|---|---|---|
| 30 | — | 1,00 | 0,92 | 0,86 |
| 40 | 1,00 | 0,94 | 0,86 | 0,80 |
| 50 | 0,92 | 0,87 | 0,82 | 0,77 |

(Tabela completa da WEG tem também 1500, 2500, 3500, 4500 e 5000 m — aqui reduzida para caber no slide.)

::: aside
Tabela **100% empírica**, do fabricante (WEG): os valores de $\alpha_{\rm{amb}}$ vêm de testes/modelos térmicos da própria WEG relacionando a capacidade de dissipação de calor real do ar (que piora com altitude e temperatura) à potência que o motor consegue entregar sem sobreaquecer. Não há uma fórmula fechada por trás — é uma tabela de engenharia, específica do fabricante.
:::

## Exemplo: Correção Ambiental

Motor de 50 cv instalado a 2000 m de altitude, com temperatura ambiente de 40 °C ($\alpha_{\rm{amb}} = 0{,}94$, pela tabela):

In [12]:
alpha_amb = 0.94   # tabela: Tamb=40C, H=2000m
Pnom = 50.0        # cv

Pdisp_amb = alpha_amb * Pnom
println("P_disp,amb = ", Pdisp_amb, " cv")

P_disp,amb = 47.0 cv


## Correção Total

Quando há partidas frequentes **e** ambiente atípico simultaneamente:

$$
P_{\rm{disp,total}} = \alpha_{\rm{pfr}} \cdot \alpha_{\rm{amb}} \cdot P_{\rm{nom}}
$$ {#eq-PdispTotal}

::: aside
Multiplicar os dois fatores de correção é uma **hipótese de independência** (simplificação prática) — assume-se que o efeito das partidas frequentes e o efeito do ambiente atípico degradam a potência disponível de forma multiplicativa e independente um do outro. Não é uma dedução termodinâmica rigorosa (na realidade os dois efeitos interagem, já que ambos afetam a mesma capacidade de dissipação de calor), mas é a aproximação padrão usada na prática de especificação.
:::

In [13]:
alpha_pfr = 0.7581   # do exemplo de partidas sucessivas
alpha_amb = 0.94     # do exemplo de correcao ambiental
Pnom = 50.0

Pdisp_total = alpha_pfr * alpha_amb * Pnom
println("P_disp,total = ", round(Pdisp_total, digits=2), " cv")

P_disp,total = 35.63 cv


## Exemplo do Texto: Necessidade de Trocar o Motor

Se a potência disponível calculada ficar **abaixo** da demandada pela carga, sobe-se para o próximo motor comercial. Do texto original: *"um motor de 40 cv aciona sem problemas uma carga de 35 cv; contudo, se a potência disponível for de 30 cv, então é necessário selecionar um motor de 50 cv."*

In [14]:
Pnom_txt, Pcarga_txt, Pdisp_txt = 40.0, 35.0, 30.0

println("Motor de 40 cv atende carga de 35 cv sem reducao? ", Pnom_txt >= Pcarga_txt)
println("Com reducao a P_disp=30 cv, ainda atende a carga de 35 cv? ", Pdisp_txt >= Pcarga_txt)
println(Pdisp_txt < Pcarga_txt ? "-> Necessario subir para o proximo motor comercial (50 cv)" : "-> OK")

Motor de 40 cv atende carga de 35 cv sem reducao? true
Com reducao a P_disp=30 cv, ainda atende a carga de 35 cv? false
-> Necessario subir para o proximo motor comercial (50 cv)


## Fator de Serviço

Se a disponibilidade ficar *ligeiramente* abaixo da carga, uma alternativa a subir de potência é usar um motor de mesma potência com **fator de serviço (F.S.) > 1**, tipicamente F.S. = 1,15.

Isso permite carga contínua de até 115% da potência nominal, sob condições especificadas — **não confundir** com a capacidade de sobrecarga momentânea (que dura apenas alguns minutos).

## Especificação de Grandezas Elétricas: Tensão

A máquina deve ser compatível com a tensão da rede. Conforme os terminais disponíveis:

::: {.incremental}
- **6 terminais:** triângulo (220 V) ou estrela (380 V);
- **9 terminais** (meio-enrolamento acessível): paralelo (220 V) ou série (440 V);
- **12 terminais** (enrolamentos totalmente separados): tripla tensão (220/380/440 V).
:::

Motor com terminais já conectados de fábrica facilita a instalação, mas perde flexibilidade de tensão.

## Exemplo: Tensão de Fase e de Linha

*Exemplo didático adicional (não consta no material original).*

Enrolamento dimensionado para 220 V de fase, ligado em estrela — qual a tensão de linha correspondente?

In [15]:
Vfase = 220.0   # V
Vlinha = Vfase * sqrt(3)
println("V_linha = ", round(Vlinha, digits=1), " V  (fase = ", Vfase, " V)")

V_linha = 381.1 V  (fase = 220.0 V)


O resultado (≈ 381 V) explica por que motores de 220/380 V são tão comuns no Brasil: o mesmo enrolamento de 220 V, ligado em estrela, já fornece a tensão de linha padrão de 380 V da rede trifásica.

## Frequência

Padrão brasileiro: 60 Hz. Abaixo da frequência nominal, reduz-se a tensão linearmente para manter o fluxo constante (evita saturação e sobrecorrente de magnetização):

$$
v(t) = -N \dfrac{d\phi(t)}{dt} \quad \rightarrow \quad \|\vec{\phi}\| = \dfrac{\|\vec{V}\|}{N \omega} = \text{constante}
$$

Acima da frequência nominal, a tensão é mantida constante (protege a isolação) e o fluxo — logo o conjugado máximo — diminui.

::: aside
Essa relação **é** dedução direta, não empírica: é a **lei de Faraday** ($v=-N\,d\phi/dt$) para uma tensão senoidal, onde $\|\vec{V}\| \propto N\omega\|\vec{\phi}\|$. Por isso manter $V/\omega$ (equivalente a $V/f$) constante é exatamente o que preserva a amplitude do fluxo magnético — nenhum ajuste empírico aqui, é física de bobinas.
:::

## Exemplo: Tensão em Frequência Reduzida (V/f Constante)

Motor com $V_{\rm{nom}} = 380$ V em $f_{\rm{nom}} = 60$ Hz, operado por inversor em $f = 45$ Hz mantendo $V/f$ constante:

In [16]:
Vnom, fnom = 380.0, 60.0
fred = 45.0

Vred = Vnom * (fred / fnom)
println("V em f = ", fred, " Hz (V/f constante) = ", round(Vred, digits=1), " V")

V em f = 45.0 Hz (V/f constante) = 285.0 V


## Corrente

Corrente nominal em plena carga; se ficar consistentemente acima, motor subdimensionado; se ficar em 1/2 a 1/3 da nominal, superdimensionado.

Na partida direta: **6 a 9 vezes** $I_{\rm{nom}}$. A norma limita a potência aparente com rotor bloqueado, em relação à potência nominal:

| $P_{\rm{n}}$ (kW) | $S_{\rm{p}}/P_{\rm{n}}$ máx. (kVA/kW) |
|---|---|
| $\leq 0{,}40$ | 22 |
| $6{,}3 < P_{\rm{n}} \leq 25$ | 12 |
| $63 < P_{\rm{n}} \leq 630$ | 10 |
| $630 < P_{\rm{n}} \leq 1600$ | 9 |

## Exemplo: Limite de Corrente de Partida

Motor com $P_n = 10$ kW (faixa $6{,}3 < P_n \leq 25$ kW $\Rightarrow S_p/P_n \leq 12$ kVA/kW):

In [17]:
Pn = 10.0          # kW
SpPn_max = 12.0    # kVA/kW, da tabela

Sp_max = SpPn_max * Pn
println("S_p maximo com rotor bloqueado = ", Sp_max, " kVA")

S_p maximo com rotor bloqueado = 120.0 kVA


## Exemplo: Sobrecorrente Ocasional

Motores com $P_{\rm{nom}} \leq 315$ kW e $V_{\rm{nom}} \leq 1$ kV devem suportar $1{,}5\,I_{\rm{nom}}$ por pelo menos 2 minutos. Para $I_{\rm{nom}} = 20$ A:

In [18]:
Inom = 20.0   # A
Iocasional = 1.5 * Inom
println("Corrente ocasional admitida (>= 2 min) = ", Iocasional, " A")

Corrente ocasional admitida (>= 2 min) = 30.0 A


## Especificação de Parâmetros Construtivos

![Vista explodida de um motor trifásico.](images/vistaExplodidaMotorW22.svg){#fig-vistaExplodida .r-stretch fig-align='center'}

## Ventilação

- **Aberto** (ODP — *Open Drip Proof*): melhor dissipação, ambientes limpos e secos;
- **Totalmente Fechado com Ventilação Externa (TFVE):**
  - **TEFC** (autoventilado) — mais comum, mas depende da velocidade do próprio eixo;
  - **TENV** (não ventilado);
  - **TEAO** (refrigeração natural);
  - **TEFV** (ventilação forçada) — indicado para baixas velocidades, onde o autoventilado esfria mal.

O tipo de ventilação define o fator $k_{\rm{v}}$ usado na @eq-PefComRepouso.

## Grau de Proteção (IP)

Norma ABNT NBR-IEC 60034-5 — duas letras (IP) + dois algarismos: 1º = corpos estranhos, 2º = água.

| 1º alg. | Indicação | 2º alg. | Indicação |
|---|---|---|---|
| 0 | Não protegida | 0 | Não protegida |
| 2 | Objetos > 12 mm | 1 | Gotejamento vertical |
| 4 | Objetos > 1,0 mm | 4 | Projeções de água |
| 5 | Poeira | 5 | Jatos de água |
| 6 | Totalmente contra poeira | 7 | Imersão temporária |

Motores abertos comuns: **IP-21, IP-23**. Motores fechados comuns: **IP-44, IP-55**. Aplicações especiais: **IP-55W, IP-56, IP-65, IP-66**.

## Carcaça e Formas Construtivas

:::: {.columns}
::: {.column width="50%"}
![Tamanho $H$ da carcaça (mm).](images/tamanhoCarcaca.svg){#fig-tamanhoCarcaca width=100% fig-align='center'}
:::
::: {.column width="50%"}
![Distribuição de temperatura no motor.](images/elevacaoTemperaturaMotor.svg){#fig-elevacaoTemperatura width=100% fig-align='center'}
:::
::::

## Formas Construtivas

![Formas construtivas de motores WEG.](images/formaConstrutiva.svg){#fig-formaConstrutiva .r-stretch fig-align='center'}

A mais comum é a **B3D**: montagem horizontal, motor com pés, eixo à direita (olhando para a caixa de ligação).

## Tolerância dos Parâmetros

A NBR 17094 permite os seguintes desvios entre valor declarado e medido:

| Grandeza | Tolerância |
|---|---|
| Rendimento ($\eta \geq 0{,}851$) | $-0{,}2\,(1-\eta)$ |
| Rendimento ($\eta < 0{,}851$) | $-0{,}15\,(1-\eta)$ |
| Fator de potência | entre $-0{,}02$ e $-0{,}07$ |
| Escorregamento ($P_{\rm{nom}} \geq 1$ kW) | $\pm 20\%$ |
| Corrente de partida | $+20\%$ |
| Torque de partida / mínimo | $-15\%$ |
| Torque máximo | $-10\%$ (mas $\geq 1{,}5\,T_{\rm{nom}}$) |
| Momento de inércia | $\pm 10\%$ |

## Exemplo: Tolerância de Rendimento

Motor com rendimento declarado $\eta = 90\%$ ($\eta \geq 0{,}851 \Rightarrow$ tolerância $-0{,}2(1-\eta)$):

In [19]:
eta = 0.90
delta_eta = -0.2 * (1 - eta)
eta_min = eta + delta_eta

println("Tolerancia = ", round(delta_eta, digits=4))
println("Rendimento medido deve ser >= ", round(eta_min, digits=4), " (", round(eta_min*100,digits=2), "%)")

Tolerancia = -0.02
Rendimento medido deve ser >= 0.88 (88.0%)


## Aspectos Econômicos

::: {.incremental}
- Especificação deve ser **numérica e objetiva** — evitar termos subjetivos como "baixas perdas" ou "elevado torque de partida";
- Especificações mais genéricas (um tipo de motor para instalação diversa) são, paradoxalmente, **mais difíceis** de elaborar que uma aplicação específica;
- Custos totais: **aquisição** + **energia** (dominante ao longo da vida útil) + **manutenção** (preventiva e corretiva);
- Projeções de custo devem considerar a vida útil de todo o **acionamento**, não só do motor.
:::

# Referências

No material original, esta seção cita principalmente:

- LOBOSCO, O. S.; DIAS, J. L. P. C. (1988);
- AUGUSTO JÚNIOR, C. (2005);
- MAMEDE FILHO, J. (2017);
- WEG — Guia de Especificação de Motores Elétricos (2021).

# Exercícios

## Exercício 1

**Um acionamento tem $J=0{,}8$ kg.m², $\omega_{\rm{op}}=150$ rad/s, dividido em 3 intervalos: $\omega=[0, 75, 150]$ rad/s com $T_{\rm{ac}}=[100, 70, 50]$ N.m. Calcule $t_{\rm{ac}}$ pelo método dos intervalos.**

. . .

In [20]:
omega = [0.0, 75.0, 150.0]
Tac   = [100.0, 70.0, 50.0]
J = 0.8

tac_total = 0.0
for k in 1:length(omega)-1
    Tbar = (Tac[k+1] + Tac[k]) / 2
    dt = J / Tbar * (omega[k+1] - omega[k])
    global tac_total += dt
end
println("t_ac total = ", round(tac_total, digits=4), " s")

t_ac total = 1.7059 s


## Exercício 2

**Um motor categoria D tem $T_{\rm{e,p}}=250$ N.m. Calcule seu torque médio eletromagnético $\overline{T}_{\rm{e}}$.**

. . .

In [21]:
Tep_D = 250.0
Tebar_D = 0.60 * Tep_D
println("T_e_bar (categoria D) = ", Tebar_D, " N.m")

T_e_bar (categoria D) = 150.0 N.m


## Exercício 3

**Uma carga com torque quadrático ($\beta=2$) tem $T_{\rm{L,p}}=20$ N.m e $T_{\rm{L,op}}=180$ N.m. Calcule $\overline{T}_{\rm{L}}$.**

. . .

In [22]:
TLp, TLop, beta = 20.0, 180.0, 2
TLbar = TLp + (TLop - TLp)/(beta+1)
println("T_L_bar = ", round(TLbar, digits=2), " N.m")

T_L_bar = 73.33 N.m


## Exercício 4

**Um ciclo de carga tem 2 patamares: 15 kW por 40 s e 30 kW por 10 s, sem repouso. Calcule a potência eficaz e compare com o pico.**

. . .

In [23]:
PLi = [15.0, 30.0]
ti  = [40.0, 10.0]
Pef = sqrt(sum(PLi.^2 .* ti) / sum(ti))
println("P_ef = ", round(Pef, digits=3), " kW (pico = ", maximum(PLi), " kW)")

P_ef = 18.974 kW (pico = 30.0 kW)


## Exercício 5

**Um motor de $P_{\rm{nom}}=25$ cv opera com $N_p=6$ partidas/hora, $t_{\rm{ac}}=5$ s e $I_p/I_{\rm{nom}}=6$. Calcule a potência disponível $P_{\rm{disp,pfr}}$.**

. . .

In [24]:
kpfr, Np, tac_h, Ip_Inom, Pnom = 1.0, 6.0, 5.0, 6.0, 25.0
alpha_pfr = sqrt((3600 - kpfr*Np*tac_h*(Ip_Inom)^2) / (3600 - 2*Np*tac_h))
println("alpha_pfr = ", round(alpha_pfr, digits=4))
println("P_disp,pfr = ", round(alpha_pfr*Pnom, digits=2), " cv")

alpha_pfr = 0.8437
P_disp,pfr = 21.09 cv


## Exercício 6

**Explique por que um motor autoventilado (TEFC) é uma má escolha para operar continuamente em baixas velocidades.**

. . .

Porque o ventilador está acoplado ao próprio eixo — em baixa velocidade, o ventilador gira devagar e a vazão de ar cai proporcionalmente, prejudicando a dissipação de calor justamente quando o motor mais precisa de refrigeração. A alternativa é usar ventilação forçada (TEFV), com fonte de ar independente da rotação do motor.

## Exercício 7

**Um motor tem rendimento declarado $\eta=80\%$ (abaixo de 0,851, tolerância $-0{,}15(1-\eta)$). Qual o rendimento mínimo aceitável medido?**

. . .

In [25]:
eta = 0.80
delta_eta = -0.15*(1-eta)
eta_min = eta + delta_eta
println("Tolerancia = ", round(delta_eta, digits=4))
println("Rendimento medido deve ser >= ", round(eta_min, digits=4))

Tolerancia = -0.03
Rendimento medido deve ser >= 0.77


## Exercício 8

**Por que a especificação técnica de um motor não deve conter termos como "baixas perdas" ou "operação livre de falha"?**

. . .

Porque são termos subjetivos, sem valor numérico verificável — não permitem ao fabricante saber exatamente o que deve ser entregue, nem ao comprador verificar se o produto atende à especificação. Uma boa especificação comunica de forma clara e numérica o que precisa ser alcançado, não como alcançá-lo.

# Obrigado!

<div style="text-align: center;">

[http://thalesmaia.com/](http://thalesmaia.com/)

</div>